# Amazon

In [ ]:
# ── Diagnostic Cell — Full flow: search → product → reviews ──────────────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, random

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

# ── Step 1: Search ─────────────────────────────────────────────────────────────
driver.get("https://www.amazon.com/")
time.sleep(3)
search = driver.find_element(By.ID, "twotabsearchtextbox")
for char in "wireless earbuds":
    search.send_keys(char)
    time.sleep(random.uniform(0.05, 0.15))
search.send_keys(Keys.ENTER)
time.sleep(4)

print("=== SEARCH RESULTS PAGE ===")
print("Title:", driver.title)
print("URL  :", driver.current_url)

# ── Step 2: Find and enter first product ───────────────────────────────────────
product_url = None
for sel in [
    "[data-component-type='s-search-result'] h2 a",
    ".s-result-item h2 a",
    "h2 a.a-link-normal",
    ".a-link-normal[href*='/dp/']",
]:
    els = driver.find_elements(By.CSS_SELECTOR, sel)
    if els:
        product_url = els[0].get_attribute("href")
        print(f"\n✅ Found product link via: '{sel}'")
        print(f"   URL: {product_url[:80]}")
        break
    else:
        print(f"❌ No match: '{sel}'")

if not product_url:
    print("\n🛑 Could not find product link — check browser for CAPTCHA")
    # Leave open for manual inspection
else:
    driver.get(product_url)
    time.sleep(4)
    if len(driver.window_handles) > 1:
        driver.switch_to.window(driver.window_handles[-1])

    print("\n=== PRODUCT PAGE ===")
    print("Title:", driver.title)
    print("URL  :", driver.current_url)

    # ── Step 3: Test product info selectors ───────────────────────────────────
    print("\n=== PRODUCT INFO SELECTORS ===")
    info_selectors = {
        "name":           ["#productTitle"],
        "brand":          ["#bylineInfo", "a#bylineInfo"],
        "price":          [".a-price .a-offscreen", "#corePriceDisplay_desktop_feature_div .a-offscreen",
                           "#price_inside_buybox", ".priceToPay .a-offscreen"],
        "rating_score":   ["#acrPopover .a-size-base.a-color-base", "span.a-icon-alt",
                           "[data-hook='rating-out-of-text']"],
        "rating_count":   ["#acrCustomerReviewText", "[data-hook='total-review-count']"],
        "availability":   ["#availability span"],
    }

    for field, sels in info_selectors.items():
        found = False
        for sel in sels:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els and els[0].text.strip():
                print(f"✅ {field:20s} via '{sel}' → '{els[0].text.strip()[:60]}'")
                found = True
                break
        if not found:
            print(f"❌ {field:20s} → no selector worked")

    # ── Step 4: Test reviews page selectors ───────────────────────────────────
    try:
        asin = driver.current_url.split("/dp/")[1].split("/")[0].split("?")[0]
    except:
        asin = driver.find_element(By.CSS_SELECTOR, "#ASIN").get_attribute("value")

    print(f"\n=== REVIEWS PAGE (ASIN: {asin}) ===")
    driver.get(f"https://www.amazon.com/product-reviews/{asin}/?sortBy=recent&pageNumber=1")
    time.sleep(4)

    print("Title:", driver.title)
    print("URL  :", driver.current_url)

    review_selectors = {
        "review card":   ["[data-hook='review']", ".review", "[id^='customer_review']"],
        "reviewer name": [".a-profile-name"],
        "star rating":   ["[data-hook='review-star-rating'] span", "[data-hook='cmps-review-star-rating'] span"],
        "review date":   ["[data-hook='review-date']"],
        "review title":  ["[data-hook='review-title'] span:not(.a-icon-alt)"],
        "review body":   ["[data-hook='review-body'] span"],
        "helpful votes": ["[data-hook='helpful-vote-statement']"],
        "next button":   ["li.a-last:not(.a-disabled) a", "li.a-last a", ".a-pagination .a-last a"],
    }

    print("\n=== REVIEW SELECTORS ===")
    for field, sels in review_selectors.items():
        found = False
        for sel in sels:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els:
                sample = els[0].text.strip()[:60] if els[0].text.strip() else "(no text)"
                print(f"✅ {field:20s} via '{sel}' → {len(els)} found | sample: '{sample}'")
                found = True
                break
        if not found:
            print(f"❌ {field:20s} → no selector worked")

    # ── Step 5: Dump first review card HTML ───────────────────────────────────
    print("\n=== FIRST REVIEW CARD HTML ===")
    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")
    if cards:
        print(cards[0].get_attribute("outerHTML")[:3000])
    else:
        print("No review cards found — dumping page body snippet:")
        print(driver.page_source[3000:6000])

print("\n✋ Browser left open — run driver.quit() when done")

In [7]:
# ── Single Cell — Amazon Product + Reviews Scraper (On-Page Reviews) ──────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from IPython.display import display
import pandas as pd
import time, json, random

# ── 1. Driver ─────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

def get_text(ctx, selector, default="N/A"):
    try: return ctx.find_element(By.CSS_SELECTOR, selector).text.strip()
    except: return default

def get_first_text(ctx, selectors, default="N/A"):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val: return val
        except: continue
    return default

def human_sleep(a=2, b=4):
    time.sleep(random.uniform(a, b))

def check_for_captcha(driver):
    if "captcha" in driver.page_source.lower() or "Type the characters" in driver.page_source:
        print("⚠️  CAPTCHA detected! Solve it in the browser then press Enter...")
        input()

try:
    # ── 2. Search ─────────────────────────────────────────────────────────────
    driver.get("https://www.amazon.com/")
    human_sleep(2, 3)
    check_for_captcha(driver)

    wait.until(EC.presence_of_element_located((By.ID, "twotabsearchtextbox")))
    search = driver.find_element(By.ID, "twotabsearchtextbox")
    search.clear()
    for char in "wireless earbuds":
        search.send_keys(char)
        time.sleep(random.uniform(0.05, 0.15))
    search.send_keys(Keys.ENTER)
    human_sleep(3, 5)
    check_for_captcha(driver)

    # ── 3. Open first product ─────────────────────────────────────────────────
    first = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")))
    product_url = first.get_attribute("href")
    driver.get(product_url)
    human_sleep(3, 5)
    check_for_captcha(driver)
    print(f"📦 {driver.title[:80]}")

    # ── 4. Scroll slowly to load all lazy content ─────────────────────────────
    for _ in range(8):
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(0.7)
    driver.execute_script("window.scrollTo(0, 0);")  # back to top
    human_sleep(1, 2)

    # ── 5. Product Info ───────────────────────────────────────────────────────
    product = {
        "url":          driver.current_url,
        "name":         get_text(driver, "#productTitle"),
        "brand":        get_text(driver, "#bylineInfo"),
        "rating_score": get_text(driver, "[data-hook='rating-out-of-text']"),
        "rating_count": get_text(driver, "#acrCustomerReviewText"),
        "price": get_first_text(driver, [
            "#corePriceDisplay_desktop_feature_div .a-price-whole",
            ".a-price[data-a-color='price'] .a-offscreen",
            "#apex_offerDisplay_desktop .a-price .a-offscreen",
            "#corePrice_feature_div .a-price .a-offscreen",
            "#price .a-price .a-offscreen",
        ]),
        "availability": get_first_text(driver, [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div span",
            "#deliveryBlockMessage span",
        ]),
        "answered_questions": get_text(driver, "#askATFLink span"),
    }

    # Highlights
    try:
        bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
        product["highlights"] = [b.text.strip() for b in bullets if b.text.strip()]
    except:
        product["highlights"] = []

    # Variants
    try:
        variant_groups = driver.find_elements(By.CSS_SELECTOR, ".a-section.a-spacing-small.poi-rotate-anchor")
        variants = {}
        for grp in variant_groups:
            label = get_text(grp, "label.a-form-label")
            opts  = [o.get_attribute("title") or o.text for o in grp.find_elements(By.CSS_SELECTOR, "li")]
            opts  = [o.strip() for o in opts if o and o.strip()]
            if label and opts:
                variants[label.rstrip(":")] = opts
        product["variants"] = variants
    except:
        product["variants"] = {}

    # Images
    try:
        thumbs = driver.find_elements(By.CSS_SELECTOR, "#altImages img")
        product["images"] = list({img.get_attribute("src") for img in thumbs if img.get_attribute("src")})
    except:
        product["images"] = []

    # Description
    try:
        desc_el = driver.find_element(By.CSS_SELECTOR, "#productDescription p, #productDescription_feature_div")
        product["description"] = desc_el.text.strip()
    except:
        product["description"] = "N/A"

    # Specs
    try:
        rows  = driver.find_elements(By.CSS_SELECTOR,
            "#productDetails_techSpec_section_1 tr, #productDetails_detailBullets_sections1 tr")
        specs = {}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                specs[cols[0].text.strip()] = cols[1].text.strip()
        if not specs:
            for item in driver.find_elements(By.CSS_SELECTOR, "#detailBullets_feature_div li"):
                text = item.text.strip()
                if ":" in text:
                    k, v = text.split(":", 1)
                    specs[k.strip().strip("\u200f")] = v.strip()
        product["specifications"] = specs
    except:
        product["specifications"] = {}

    print(f"✅ Name    : {product['name'][:70]}")
    print(f"   Price  : {product['price']}")
    print(f"   Rating : {product['rating_score']} — {product['rating_count']}")

    # ── 6. Scrape reviews directly on the product page ────────────────────────
    # Amazon shows ~8 reviews in the "Top reviews" section without login
    all_reviews = []

    try:
        review_section = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "[data-hook='review'], #cm-cr-dp-review-list")
        ))
    except:
        print("⚠️  Review section not found, scrolling more...")
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(0.8)

    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")
    print(f"\n📝 Found {len(cards)} reviews on product page")

    for card in cards:
        def g(sel, default="N/A"):
            try: return card.find_element(By.CSS_SELECTOR, sel).text.strip()
            except: return default

        # Star rating
        try:
            star_el = card.find_element(By.CSS_SELECTOR,
                "[data-hook='review-star-rating'] span, [data-hook='cmps-review-star-rating'] span")
            rating = star_el.get_attribute("innerHTML").split(" ")[0]
        except:
            rating = "N/A"

        # Review images
        try:
            imgs   = card.find_elements(By.CSS_SELECTOR, "[data-hook='review-image-tile']")
            images = [i.get_attribute("src") for i in imgs if i.get_attribute("src")]
        except:
            images = []

        all_reviews.append({
            "reviewer":      g(".a-profile-name"),
            "rating":        rating,
            "date":          g("[data-hook='review-date']"),
            "verified":      g("[data-hook='avp-badge']"),
            "title":         g("h5[data-hook='reviewTitle']"),
            "body":          g("p span, [data-hook='review-body'] p span, [data-hook='review-body'] p"),
            "helpful_votes": g("[data-hook='helpful-vote-statement']"),
            "variant":       g("[data-hook='format-strip']"),
            "image_count":   len(images),
            "images":        images,
        })

    print(f"🎉 Scraped {len(all_reviews)} reviews")

    # ── 7. Save ───────────────────────────────────────────────────────────────
    final = {**product, "reviews": all_reviews}
    with open("amazon_product.json", "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)

    df = pd.DataFrame(all_reviews).drop(columns=["images"], errors="ignore")
    df.to_csv("amazon_reviews.csv", index=False)
    print("💾 Saved: amazon_product.json  |  amazon_reviews.csv")
    display(df)

except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()
    print(f"\n🔍 URL  : {driver.current_url}")
    print(f"🔍 Title: {driver.title}")

finally:
    time.sleep(2)
    driver.quit()
    print("🛑 Driver closed")

📦 Amazon.com: HAOYUYAN Wireless Earbuds, Sports Bluetooth Headphones, 80Hrs Playti
✅ Name    : HAOYUYAN Wireless Earbuds, Sports Bluetooth Headphones, 80Hrs Playtime
   Price  : 1,404
   Rating : 4.5 out of 5 — (1,720)

📝 Found 13 reviews on product page
🎉 Scraped 13 reviews
💾 Saved: amazon_product.json  |  amazon_reviews.csv


,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count
0,Magdalena Tzep,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Comfortable and great for workouts,I bought these wireless sports earbuds and the...,One person found this helpful,Color: Black,0
1,Yadira Viguera,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Excellent value for everyday use 🎧,The HAOYUYAN earbuds pleasantly surprised me w...,N/A,Color: Black,0
2,Nay,5,"Reviewed in the United States on April 26, 2026",Verified Purchase,Budget-Friendly Hearing Aids with Decent Perfo...,"For the price, these hearing aids are a reason...",N/A,Color: Rose gold,0
3,Ce,4,"Reviewed in the United States on April 23, 2026",Verified Purchase,"Good fit, nice sound, great price",These are great! I love that they're pink. The...,One person found this helpful,Color: A-Rose Gold,0
4,sandra,5,"Reviewed in the United States on April 26, 2026",Verified Purchase,Good quality!,These wireless earbuds are a great budget-frie...,One person found this helpful,Color: Black,2
5,Nayamí Anet,5,"Reviewed in the United States on April 22, 2026",Verified Purchase,Really cool,"Comfortable, Great Sound, and Perfect for Work...",3 people found this helpful,Color: GrayBlack,2
6,Natali almeida,5,"Reviewed in the United States on April 25, 2026",Verified Purchase,Auriculares inalambricos,These headphones surprised me a lot. They are ...,N/A,Color: Black,1
7,Oscar,5,"Reviewed in the United States on April 19, 2026",Verified Purchase,I love,They are the best in headphones economy qualit...,One person found this helpful,Color: Rose gold,0
8,Massimo Croci,5,"Reviewed in Ireland on November 12, 2025",Verified Purchase,Wonderful,Earbuds work perfectly !!!! Clear sound during...,N/A,N/A,2
9,Roxy,5,"Reviewed in Japan on November 22, 2025",Verified Purchase,"100% recommended on price, look and quality.",I bought received this 2 days ago and took it ...,N/A,N/A,0


🛑 Driver closed


In [ ]:
# ── Single Cell — Amazon Search Results + Per-Product Reviews Scraper ─────────
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from IPython.display import display
import pandas as pd
import time, json, random

# ── 1. Driver ─────────────────────────────────────────────────────────────────
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})
wait = WebDriverWait(driver, 20)

def get_text(ctx, selector, default="N/A"):
    try: return ctx.find_element(By.CSS_SELECTOR, selector).text.strip()
    except: return default

def get_first_text(ctx, selectors, default="N/A"):
    for sel in selectors:
        try:
            val = ctx.find_element(By.CSS_SELECTOR, sel).text.strip()
            if val: return val
        except: continue
    return default

def human_sleep(a=2, b=4):
    time.sleep(random.uniform(a, b))

def check_for_captcha(driver):
    if "captcha" in driver.page_source.lower() or "Type the characters" in driver.page_source:
        print("⚠️  CAPTCHA detected! Solve it in the browser then press Enter...")
        input()

def scrape_product(driver, url, index):
    """Navigate to a product page and scrape all info + on-page reviews."""
    driver.get(url)
    human_sleep(2, 4)
    check_for_captcha(driver)

    # Scroll to load lazy content
    for _ in range(8):
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(0.5)
    driver.execute_script("window.scrollTo(0, 0);")
    human_sleep(1, 2)

    product = {
        "index":        index,
        "url":          driver.current_url,
        "name":         get_text(driver, "#productTitle"),
        "brand":        get_text(driver, "#bylineInfo"),
        "rating_score": get_text(driver, "[data-hook='rating-out-of-text']"),
        "rating_count": get_text(driver, "#acrCustomerReviewText"),
        "price": get_first_text(driver, [
            "#corePriceDisplay_desktop_feature_div .a-price-whole",
            ".a-price[data-a-color='price'] .a-offscreen",
            "#apex_offerDisplay_desktop .a-price .a-offscreen",
            "#corePrice_feature_div .a-price .a-offscreen",
            "#price .a-price .a-offscreen",
        ]),
        "availability": get_first_text(driver, [
            "#availability span",
            "#availabilityInsideBuyBox_feature_div span",
            "#deliveryBlockMessage span",
        ]),
        "answered_questions": get_text(driver, "#askATFLink span"),
    }

    # Highlights
    try:
        bullets = driver.find_elements(By.CSS_SELECTOR, "#feature-bullets li span.a-list-item")
        product["highlights"] = [b.text.strip() for b in bullets if b.text.strip()]
    except:
        product["highlights"] = []

    # Variants
    try:
        variant_groups = driver.find_elements(By.CSS_SELECTOR, ".a-section.a-spacing-small.poi-rotate-anchor")
        variants = {}
        for grp in variant_groups:
            label = get_text(grp, "label.a-form-label")
            opts  = [o.get_attribute("title") or o.text for o in grp.find_elements(By.CSS_SELECTOR, "li")]
            opts  = [o.strip() for o in opts if o and o.strip()]
            if label and opts:
                variants[label.rstrip(":")] = opts
        product["variants"] = variants
    except:
        product["variants"] = {}

    # Images
    try:
        thumbs = driver.find_elements(By.CSS_SELECTOR, "#altImages img")
        product["images"] = list({img.get_attribute("src") for img in thumbs if img.get_attribute("src")})
    except:
        product["images"] = []

    # Description
    try:
        desc_el = driver.find_element(By.CSS_SELECTOR, "#productDescription p, #productDescription_feature_div")
        product["description"] = desc_el.text.strip()
    except:
        product["description"] = "N/A"

    # Specs
    try:
        rows  = driver.find_elements(By.CSS_SELECTOR,
            "#productDetails_techSpec_section_1 tr, #productDetails_detailBullets_sections1 tr")
        specs = {}
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 2:
                specs[cols[0].text.strip()] = cols[1].text.strip()
        if not specs:
            for item in driver.find_elements(By.CSS_SELECTOR, "#detailBullets_feature_div li"):
                text = item.text.strip()
                if ":" in text:
                    k, v = text.split(":", 1)
                    specs[k.strip().strip("\u200f")] = v.strip()
        product["specifications"] = specs
    except:
        product["specifications"] = {}

    # ── Reviews (on-page only, no navigation) ─────────────────────────────────
    reviews = []
    cards = driver.find_elements(By.CSS_SELECTOR, "[data-hook='review']")

    for card in cards:
        def g(sel, default="N/A"):
            try: return card.find_element(By.CSS_SELECTOR, sel).text.strip()
            except: return default

        try:
            star_el = card.find_element(By.CSS_SELECTOR,
                "[data-hook='review-star-rating'] span, [data-hook='cmps-review-star-rating'] span")
            rating = star_el.get_attribute("innerHTML").split(" ")[0]
        except:
            rating = "N/A"

        try:
            imgs   = card.find_elements(By.CSS_SELECTOR, "[data-hook='review-image-tile']")
            images = [i.get_attribute("src") for i in imgs if i.get_attribute("src")]
        except:
            images = []

        reviews.append({
            "reviewer":      g(".a-profile-name"),
            "rating":        rating,
            "date":          g("[data-hook='review-date']"),
            "verified":      g("[data-hook='avp-badge']"),
            "title":         g("h5[data-hook='reviewTitle']"),
            "body":          g("span[data-hook='review-body']"),
            "helpful_votes": g("[data-hook='helpful-vote-statement']"),
            "variant":       g("[data-hook='format-strip']"),
            "image_count":   len(images),
            "images":        images,
        })

    product["reviews"] = reviews
    return product

try:
    # ── 2. Search ─────────────────────────────────────────────────────────────
    driver.get("https://www.amazon.com/")
    human_sleep(2, 3)
    check_for_captcha(driver)

    wait.until(EC.presence_of_element_located((By.ID, "twotabsearchtextbox")))
    search = driver.find_element(By.ID, "twotabsearchtextbox")
    search.clear()
    for char in "wireless earbuds":
        search.send_keys(char)
        time.sleep(random.uniform(0.05, 0.15))
    search.send_keys(Keys.ENTER)
    human_sleep(3, 5)
    check_for_captcha(driver)

    # ── 3. Collect all product URLs from search results page ──────────────────
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")))

    # Scroll results page to load all cards
    for _ in range(5):
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(0.5)

    # Grab unique /dp/ links only (filters out nav/ad junk)
    all_links = driver.find_elements(By.CSS_SELECTOR, ".a-link-normal[href*='/dp/']")
    product_urls = []
    seen = set()
    for el in all_links:
        href = el.get_attribute("href") or ""
        # Extract clean base URL up to /dp/ASIN
        if "/dp/" in href:
            try:
                asin  = href.split("/dp/")[1].split("/")[0].split("?")[0]
                clean = f"https://www.amazon.com/dp/{asin}"
                if asin not in seen and len(asin) == 10:  # ASINs are always 10 chars
                    seen.add(asin)
                    product_urls.append(clean)
            except:
                continue

    print(f"🔍 Found {len(product_urls)} unique products on results page")
    for i, u in enumerate(product_urls):
        print(f"   {i+1}. {u}")

    # ── 4. Scrape each product ─────────────────────────────────────────────────
    search_results_url = driver.current_url  # save to go back if needed
    all_products = []

    for i, url in enumerate(product_urls):
        print(f"\n{'─'*60}")
        print(f"📦 [{i+1}/{len(product_urls)}] Scraping: {url}")
        try:
            product = scrape_product(driver, url, i + 1)
            all_products.append(product)
            print(f"   ✅ {product['name'][:60]}")
            print(f"   💬 {len(product['reviews'])} reviews scraped")
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            all_products.append({"index": i+1, "url": url, "error": str(e), "reviews": []})

        human_sleep(2, 4)  # be polite between requests

    # ── 5. Save ───────────────────────────────────────────────────────────────
    with open("amazon_all_products.json", "w", encoding="utf-8") as f:
        json.dump(all_products, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Saved: amazon_all_products.json")

    # Flat reviews CSV — each row = one review, with product name column
    all_reviews_flat = []
    for p in all_products:
        for r in p.get("reviews", []):
            all_reviews_flat.append({
                "product_index": p.get("index"),
                "product_name":  p.get("name", "N/A"),
                "product_url":   p.get("url", "N/A"),
                **{k: v for k, v in r.items() if k != "images"},
            })

    df = pd.DataFrame(all_reviews_flat)
    df.to_csv("amazon_all_reviews.csv", index=False)
    print(f"💾 Saved: amazon_all_reviews.csv ({len(df)} total reviews)")

    # Summary table
    summary = pd.DataFrame([{
        "index":        p.get("index"),
        "name":         p.get("name", "N/A")[:50],
        "price":        p.get("price", "N/A"),
        "rating_score": p.get("rating_score", "N/A"),
        "rating_count": p.get("rating_count", "N/A"),
        "reviews_scraped": len(p.get("reviews", [])),
    } for p in all_products])

    print(f"\n📊 Summary ({len(all_products)} products):")
    display(summary)
    display(df.head(10))

except Exception as e:
    import traceback
    print(f"❌ Error: {e}")
    traceback.print_exc()
    print(f"\n🔍 URL  : {driver.current_url}")
    print(f"🔍 Title: {driver.title}")

finally:
    time.sleep(2)
    driver.quit()
    print("🛑 Driver closed")

🔍 Found 52 unique products on results page
   1. https://www.amazon.com/dp/B0BTYCRJSS
   2. https://www.amazon.com/dp/B0BTYDLTM3
   3. https://www.amazon.com/dp/B0D5HSP346
   4. https://www.amazon.com/dp/B0D5HN8B1F
   5. https://www.amazon.com/dp/B0BTYCJXBK
   6. https://www.amazon.com/dp/B0GHM7LCBV
   7. https://www.amazon.com/dp/B0G2LV6439
   8. https://www.amazon.com/dp/B0GMHCVW6H
   9. https://www.amazon.com/dp/B0GMH4LKWV
   10. https://www.amazon.com/dp/B0GMBXPRCF
   11. https://www.amazon.com/dp/B0GMH4WPB3
   12. https://www.amazon.com/dp/B0GQS7SKRQ
   13. https://www.amazon.com/dp/B0GTV8KLZQ
   14. https://www.amazon.com/dp/B0GF8K5QBN
   15. https://www.amazon.com/dp/B0GHZ4X4FG
   16. https://www.amazon.com/dp/B0GHZ23HY3
   17. https://www.amazon.com/dp/B0CRTR3PMF
   18. https://www.amazon.com/dp/B0CRTYZG5C
   19. https://www.amazon.com/dp/B0CRTPF7CZ
   20. https://www.amazon.com/dp/B0CRTF579W
   21. https://www.amazon.com/dp/B0CRTN7C8C
   22. https://www.amazon.com/dp/B0DGJ7HYG

,index,name,price,rating_score,rating_count,reviews_scraped
0,1,"Soundcore by Anker P20i True Wireless Earbuds,...","1,227",4.4 out of 5,"(103,631)",11
1,2,"Soundcore by Anker P20i True Wireless Earbuds,...","1,293",4.4 out of 5,"(103,631)",11
2,3,"Soundcore by Anker P20i True Wireless Earbuds,...","1,535",4.4 out of 5,"(103,631)",11
3,4,"Soundcore by Anker P20i True Wireless Earbuds,...","1,539",4.4 out of 5,"(103,631)",11
4,5,"Soundcore by Anker P20i True Wireless Earbuds,...","1,437",4.4 out of 5,"(103,631)",11
5,6,"HAOYUYAN Wireless Earbuds, Sports Bluetooth He...","1,399",4.5 out of 5,"(1,646)",13
6,7,"HAOYUYAN Wireless Earbuds, Sports Bluetooth He...",N/A,4.5 out of 5,"(1,646)",13
7,8,"HAOYUYAN Wireless Earbuds, Bluetooth Headphone...","1,657",4.5 out of 5,"(1,646)",13
8,9,HAOYUYAN Wireless Earbuds Bluetooth 5.3 Headph...,"1,231",4.5 out of 5,"(1,646)",13
9,10,HAOYUYAN Wireless Earbuds Bluetooth 5.3 Headph...,"1,657",4.5 out of 5,"(1,646)",13


,product_index,product_name,product_url,reviewer,rating,date,verified,title,body,helpful_votes,variant,image_count
0,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,From NJ,5.0,"Reviewed in the United States on April 17, 2026",Verified Purchase,N/A,"Outstanding! For the low price (~$20), this is...",9 people found this helpful,N/A,0
1,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,jack,5.0,"Reviewed in the United States on April 17, 2026",Verified Purchase,N/A,"I never write reviews, but I have to for this ...",6 people found this helpful,N/A,0
2,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Brian D Luna,5.0,"Reviewed in the United States on April 17, 2026",Verified Purchase,N/A,Anker makes really good stuff all the time. Fo...,4 people found this helpful,N/A,0
3,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Suwarna,4.0,"Reviewed in the United States on March 26, 2026",Verified Purchase,N/A,"I bought two of these, They Worked Great but a...",8 people found this helpful,N/A,1
4,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Lenin Goud,5.0,"Reviewed in the United States on April 6, 2026",Verified Purchase,N/A,These earbuds are impressive for the price. Th...,4 people found this helpful,N/A,0
5,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Severina,5.0,"Reviewed in the United States on February 22, ...",Verified Purchase,N/A,These earbuds get a true 5 stars for multiple ...,43 people found this helpful,N/A,0
6,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Tfletch22222,5.0,"Reviewed in Australia on March 11, 2026",Verified Purchase,N/A,Excellent product. Great sound. Easy to connec...,N/A,N/A,0
7,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Rinky Nakwal,5.0,"Reviewed in India on August 5, 2025",Verified Purchase,N/A,Excellent quality sound with advance dedicated...,N/A,N/A,0
8,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,Renadah,5.0,"Reviewed in Egypt on February 6, 2026",Verified Purchase,N/A,ممتاز جدا و العلبه مقفوله,N/A,N/A,0
9,1,"Soundcore by Anker P20i True Wireless Earbuds,...",https://www.amazon.com/dp/B0BTYCRJSS?th=1,xavi,5.0,"Reviewed in Spain on April 22, 2026",Verified Purchase,N/A,"En mi opinión, son los mejores que te puedes c...",N/A,N/A,0


🛑 Driver closed
